In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import urllib.request
import ssl
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm

# ==========================================
# 1. 基础配置 (Configuration)
# ==========================================

class Config:
    """实验超参数配置"""
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 模型参数
        self.embed_dim = 64
        self.seq_len = 100
        self.tcan_layers = 2
        self.kernel_size = 3
        self.dropout = 0.3

        # 训练参数
        self.batch_size = 64
        self.epochs = 20
        self.lr = 0.001
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # 数据集路径
        self.data_dir = "./data"
        self.dataset_name = "assistment-2009-2010-skill"
        # 备用链接 (如果 EduData 失败，尝试此链接)
        self.fallback_url = "https://raw.githubusercontent.com/bigdata-ustc/EduData/master/static/assistments/skill_builder_data.csv"

# ==========================================
# 2. 数据处理模块 (Data Processing)
# ==========================================

class AssistDataset(Dataset):
    def __init__(self, q_ids, labels, s_ids, max_len):
        self.q_ids = q_ids
        self.labels = labels
        self.s_ids = s_ids
        self.max_len = max_len

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        q_seq = self.q_ids[idx]
        r_seq = self.labels[idx]
        s_id = self.s_ids[idx]
        seq_len = len(q_seq)

        if seq_len >= self.max_len:
            q_seq, r_seq = q_seq[-self.max_len:], r_seq[-self.max_len:]
            mask = np.ones(self.max_len, dtype=np.float32)
        else:
            pad_len = self.max_len - seq_len
            q_seq = np.pad(q_seq, (pad_len, 0), 'constant', constant_values=0)
            r_seq = np.pad(r_seq, (pad_len, 0), 'constant', constant_values=0)
            mask = np.concatenate([np.zeros(pad_len), np.ones(seq_len)]).astype(np.float32)

        return (
            torch.tensor(q_seq, dtype=torch.long),
            torch.tensor(r_seq, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(s_id, dtype=torch.long)
        )

class DataProcessor:
    def __init__(self, config):
        self.config = config
        if not os.path.exists(config.data_dir):
            os.makedirs(config.data_dir)

    def _install_edudata(self):
        """自动安装 EduData 库"""
        print("正在安装 EduData 库以获取数据集...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
            return True
        except Exception as e:
            print(f"EduData 安装失败: {e}")
            return False

    def download_data(self):
        """优先使用 EduData 下载，失败则使用备用 URL"""
        csv_file = self._find_csv(self.config.data_dir)
        if csv_file:
            print(f"发现本地数据文件: {csv_file}")
            return csv_file

        print(f"尝试使用 EduData 获取数据集: {self.config.dataset_name}")
        try:
            from EduData import get_data
            get_data(self.config.dataset_name, self.config.data_dir)
        except ImportError:
            if self._install_edudata():
                try:
                    from EduData import get_data
                    get_data(self.config.dataset_name, self.config.data_dir)
                except Exception as e:
                    print(f"EduData 下载报错: {e}")
            else:
                print("跳过 EduData，尝试手动下载...")

        # 再次检查是否下载成功
        csv_file = self._find_csv(self.config.data_dir)
        if csv_file: return csv_file

        # Fallback 方案
        print(f"正在尝试从备用链接下载: {self.config.fallback_url}")
        save_path = os.path.join(self.config.data_dir, "skill_builder_data.csv")
        context = ssl._create_unverified_context()
        try:
            req = urllib.request.Request(self.config.fallback_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, context=context) as response, open(save_path, 'wb') as out_file:
                out_file.write(response.read())
            return save_path
        except Exception as e:
            raise RuntimeError(f"所有下载方式均失败。报错信息: {e}")

    def _find_csv(self, root_dir):
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.endswith(".csv") and "skill" in file.lower():
                    return os.path.join(root, file)
        return None

    def process(self, order_strategy="chronological"):
        file_path = self.download_data()
        print("读取并清洗数据...")
        try:
            df = pd.read_csv(file_path, encoding='latin-1', low_memory=False)
        except:
            df = pd.read_csv(file_path, encoding='utf-8', low_memory=False)

        user_col, item_col, correct_col, order_col = 'user_id', 'problem_id', 'correct', 'order_id'
        skill_col = next((c for c in df.columns if 'skill_id' in c), 'skill_id')
        df.dropna(subset=[skill_col], inplace=True)

        user_map = {val: i+1 for i, val in enumerate(df[user_col].unique())}
        prob_map = {val: i+1 for i, val in enumerate(df[item_col].unique())}
        skill_map = {val: i+1 for i, val in enumerate(df[skill_col].unique())}

        df['user_id_mapped'] = df[user_col].map(user_map)
        df['item_id'] = df[item_col].map(prob_map)
        df['skill_id'] = df[skill_col].map(skill_map)

        self.config.num_students = len(user_map) + 1
        self.config.num_questions = len(prob_map) + 1
        self.config.num_concepts = len(skill_map) + 1

        item_difficulty = df.groupby('item_id')[correct_col].mean()
        diff_values = np.zeros(self.config.num_questions)
        for iid, acc in item_difficulty.items():
            diff_values[iid] = 1.0 - acc
        df['difficulty'] = df['item_id'].map(item_difficulty)

        q_k_relation = np.zeros((self.config.num_questions, self.config.num_concepts))
        tmp_df = df[['item_id', 'skill_id']].drop_duplicates()
        for _, row in tmp_df.iterrows():
            q_k_relation[int(row['item_id']), int(row['skill_id'])] = 1
        row_sums = q_k_relation.sum(axis=1)
        row_sums[row_sums == 0] = 1
        q_k_relation = q_k_relation / row_sums[:, np.newaxis]

        all_q, all_r, all_s = [], [], []
        users = df['user_id_mapped'].unique()[:1500]
        df_small = df[df['user_id_mapped'].isin(users)]

        for uid, group in tqdm(df_small.groupby('user_id_mapped'), desc=f"Strategy: {order_strategy}"):
            if len(group) < 5: continue

            if order_strategy == "chronological":
                group = group.sort_values(by=order_col)
            elif order_strategy == "reverse":
                group = group.sort_values(by=order_col, ascending=False)
            elif order_strategy == "random":
                group = group.sample(frac=1)
            elif order_strategy == "difficulty_asc":
                group = group.sort_values(by='difficulty')
            elif order_strategy == "skill_grouped":
                group = group.sort_values(by=['skill_id', order_col])

            all_q.append(group['item_id'].values)
            all_r.append(group[correct_col].values)
            all_s.append(uid)

        return all_q, all_r, all_s, q_k_relation, diff_values

# ==========================================
# 3. 模型定义与实验逻辑 (保持不变)
# ==========================================

class HeteroGraphEmbedding(nn.Module):
    def __init__(self, config, diff_values):
        super(HeteroGraphEmbedding, self).__init__()
        self.question_emb = nn.Embedding(config.num_questions, config.embed_dim, padding_idx=0)
        self.concept_emb = nn.Embedding(config.num_concepts, config.embed_dim, padding_idx=0)
        self.register_buffer('diff_values', torch.tensor(diff_values, dtype=torch.float32))
        self.diff_proj = nn.Linear(1, config.embed_dim)

    def forward(self, question_ids, q_k_relation):
        k_emb_weight = self.concept_emb.weight
        q_k_agg = torch.matmul(q_k_relation, k_emb_weight)
        d_val = self.diff_values[question_ids].unsqueeze(-1)
        d_feat = self.diff_proj(d_val)
        q_id_feat = self.question_emb(question_ids)
        q_k_feat = F.embedding(question_ids, q_k_agg, padding_idx=0)
        return q_id_feat + q_k_feat + d_feat

class TemporalAttention(nn.Module):
    def __init__(self, embed_dim, dropout=0.1):
        super(TemporalAttention, self).__init__()
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, L, D = x.size()
        Q, K, V = self.query(x), self.key(x), self.value(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)
        mask = torch.tril(torch.ones(L, L, device=x.device))
        scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(self.dropout(attn), V)

class TCANBlock(nn.Module):
    def __init__(self, embed_dim, kernel_size, dilation, dropout):
        super(TCANBlock, self).__init__()
        self.ta = TemporalAttention(embed_dim, dropout)
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(embed_dim, embed_dim, kernel_size, dilation=dilation, padding=self.padding)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        z = self.ta(x)
        conv_out = self.conv(z.permute(0, 2, 1))[:, :, :z.size(1)].permute(0, 2, 1)
        return self.dropout(self.relu(self.norm(conv_out + residual)))

class HIG_TCAN_CD_Model(nn.Module):
    def __init__(self, config, q_k_relation, diff_values):
        super(HIG_TCAN_CD_Model, self).__init__()
        self.config = config
        self.register_buffer('q_k_relation', torch.tensor(q_k_relation, dtype=torch.float32))
        self.graph_emb = HeteroGraphEmbedding(config, diff_values)
        self.student_emb = nn.Embedding(config.num_students, config.embed_dim, padding_idx=0)
        self.input_proj = nn.Linear(config.embed_dim + 1, config.embed_dim)
        self.tcan_layers = nn.ModuleList([
            TCANBlock(config.embed_dim, config.kernel_size, 2**i, config.dropout)
            for i in range(config.tcan_layers)
        ])
        self.pred_mlp = nn.Sequential(
            nn.Linear(config.embed_dim * 3, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, q_ids, answers, student_ids):
        q_emb = self.graph_emb(q_ids, self.q_k_relation)
        x = self.input_proj(torch.cat([q_emb, answers.unsqueeze(-1)], dim=-1))
        h = x
        for layer in self.tcan_layers: h = layer(h)
        s_static = self.student_emb(student_ids).unsqueeze(1).expand(-1, q_ids.size(1), -1)
        return h, q_emb, s_static

def train_and_eval(config, data_bundle):
    train_loader, test_loader, q_k_rel, diff_values = data_bundle
    model = HIG_TCAN_CD_Model(config, q_k_rel, diff_values).to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

    best_auc = 0
    for epoch in range(config.epochs):
        model.train()
        for q, r, mask, s in train_loader:
            q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
            optimizer.zero_grad()
            h_seq, q_seq_emb, s_static = model(q, r, s)
            feat = torch.cat([h_seq[:, :-1, :], q_seq_emb[:, 1:, :], s_static[:, 1:, :]], dim=-1)
            preds = model.pred_mlp(feat).squeeze(-1)
            loss = (F.binary_cross_entropy(preds, r[:, 1:], reduction='none') * mask[:, 1:]).sum() / (mask[:, 1:].sum() + 1e-8)
            loss.backward()
            optimizer.step()

        model.eval()
        all_p, all_t = [], []
        with torch.no_grad():
            for q, r, mask, s in test_loader:
                q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
                h, qe, ss = model(q, r, s)
                feat = torch.cat([h[:, :-1, :], qe[:, 1:, :], ss[:, 1:, :]], dim=-1)
                p = model.pred_mlp(feat).squeeze(-1)
                m = mask[:, 1:].bool()
                all_p.extend(p[m].cpu().numpy())
                all_t.extend(r[:, 1:][m].cpu().numpy())

        auc = roc_auc_score(all_t, all_p)
        if auc > best_auc: best_auc = auc

    return best_auc

def run_extended_ablation():
    config = Config()
    processor = DataProcessor(config)
    results = {}

    strategies = ["chronological", "reverse", "random", "difficulty_asc", "skill_grouped"]

    for strategy in strategies:
        print(f"\n>>> 运行实验: 序列顺序 = {strategy}")
        q_seqs, r_seqs, s_seqs, q_k_rel, diff_values = processor.process(order_strategy=strategy)
        t_q, v_q, t_r, v_r, t_s, v_s = train_test_split(q_seqs, r_seqs, s_seqs, test_size=0.2, random_state=42)
        train_loader = DataLoader(AssistDataset(t_q, t_r, t_s, config.seq_len), batch_size=config.batch_size, shuffle=True)
        test_loader = DataLoader(AssistDataset(v_q, v_r, v_s, config.seq_len), batch_size=config.batch_size)

        best_auc = train_and_eval(config, (train_loader, test_loader, q_k_rel, diff_values))
        results[strategy] = best_auc
        print(f"--- 实验结果 ({strategy}): Best AUC = {best_auc:.4f} ---")

    print("\n" + "="*40)
    print("TCAN 序列顺序消融实验总结")
    print("-" * 40)
    for k in sorted(results, key=results.get, reverse=True):
        print(f"{k:<20} | {results[k]:.4f}")
    print("="*40)

if __name__ == "__main__":
    run_extended_ablation()


>>> 运行实验: 序列顺序 = chronological
尝试使用 EduData 获取数据集: assistment-2009-2010-skill
正在安装 EduData 库以获取数据集...


downloader, INFO http://base.ustc.edu.cn/data/ASSISTment/2009_skill_builder_data_corrected.zip is saved as data/2009_skill_builder_data_corrected.zip


downloader, INFO data/2009_skill_builder_data_corrected.zip is unzip to data/2009_skill_builder_data_corrected



读取并清洗数据...


Strategy: chronological: 100%|██████████| 1500/1500 [00:01<00:00, 1304.98it/s]


...

--- 实验结果 (random): Best AUC = 0.7517 ---

>>> 运行实验: 序列顺序 = difficulty_asc
发现本地数据文件: ./data/2009_skill_builder_data_corrected/skill_builder_data_corrected.csv
读取并清洗数据...


Strategy: difficulty_asc: 100%|██████████| 1500/1500 [00:00<00:00, 1893.85it/s]


--- 实验结果 (difficulty_asc): Best AUC = 0.8439 ---

>>> 运行实验: 序列顺序 = skill_grouped
发现本地数据文件: ./data/2009_skill_builder_data_corrected/skill_builder_data_corrected.csv
读取并清洗数据...


Strategy: skill_grouped: 100%|██████████| 1500/1500 [00:01<00:00, 1011.30it/s]


--- 实验结果 (skill_grouped): Best AUC = 0.7790 ---

TCAN 序列顺序消融实验总结
----------------------------------------
difficulty_asc       | 0.8439
chronological        | 0.8219
reverse              | 0.8131
skill_grouped        | 0.7790
random               | 0.7517
